In [1]:
import numpy as np
import os
import pandas as pd

In [2]:
gtf_fn = "/groups/cgsd/xianjie/data/refseq/refdata-cellranger-GRCh38-3.0.0/genes/genes.gtf"
out_dir = "../"

In [3]:
def extract_genes(fn, out_fn):
    in_fp = open(fn, "r")
    out_fp = open(out_fn, "w")
    
    header = "\t".join(["gene_id", "chrom", "start", "end", "strand", "gene_name", "hgnc_id", "gene_type"])
    out_fp.write(header + "\n")
    
    nl = 0
    for line in in_fp:
        nl += 1
        if line.startswith("#"):
            continue
        items = line.strip().split("\t")
        if len(items) != 9:
            print("[E] n_fields != 9 in line %d." % nl)
            break
        chrom, source, biotype, start, end, score, strand, frame, attribute = items[:9]
        if biotype != "gene":
            continue
        if strand not in ("+", "-"):
            print("[E] invalid strand '%s' in line %d." % (strand, nl))
            break
        attr_list = [attr.strip() for attr in attribute.split(";") if len(attr) > 0]
        gene_id = gene_name = gene_type = hgnc_id = ""
        for attr in attr_list:
            items_attr = [s.strip('"') for s in attr.split(" ") if len(s) > 0]
            if len(items_attr) != 2:
                print("[E] invalid attribute in line %d." % nl)
            key, value = items_attr[:2]
            if key == "gene_id":
                gene_id = value
            elif key == "gene_name":
                gene_name = value
            elif key == "gene_biotype":
                gene_type = value
            elif key == "hgnc_id":
                hgnc_id = value
        rec = "\t".join([gene_id, chrom, start, end, strand, gene_name, hgnc_id, gene_type])
        out_fp.write(rec + "\n")
        
    in_fp.close()
    out_fp.close()

In [4]:
tsv_fn = os.path.join(out_dir, "refdata-cellranger-GRCh38-3.0.0.all.8column.genes.unsort.tsv")
extract_genes(gtf_fn, tsv_fn)

In [5]:
dat = pd.read_csv(tsv_fn, sep = "\t")
dat

,gene_id,chrom,start,end,strand,gene_name,hgnc_id,gene_type
0,ENSG00000243485,1,29554,31109,+,MIR1302-2HG,NaN,lincRNA
1,ENSG00000237613,1,34554,36081,-,FAM138A,NaN,lincRNA
2,ENSG00000186092,1,65419,71585,+,OR4F5,NaN,protein_coding
3,ENSG00000238009,1,89295,133723,-,AL627309.1,NaN,lincRNA
4,ENSG00000239945,1,89551,91105,-,AL627309.3,NaN,lincRNA
...,...,...,...,...,...,...,...,...
33533,ENSG00000277856,KI270726.1,26241,26534,+,AC233755.2,NaN,protein_coding
33534,ENSG00000275063,KI270726.1,41444,41876,+,AC233755.1,NaN,protein_coding
33535,ENSG00000271254,KI270711.1,4612,29626,-,AC240274.1,NaN,protein_coding
33536,ENSG00000277475,KI270713.1,31698,32528,-,AC213203.1,NaN,protein_coding


### Check N.A.

In [6]:
for column in dat.columns:
    n_na = np.sum(dat[column].isna())
    print(column, n_na)

gene_id 0
chrom 0
start 0
end 0
strand 0
gene_name 0
hgnc_id 33538
gene_type 0


### Check order

In [7]:
uniq_chroms = np.unique(dat["chrom"])
for chrom in uniq_chroms:
    start = dat["start"][dat["chrom"] == chrom].values
    if len(start) == 1:
        print(chrom, 1, True)
    start_pre = start[:-1]
    start_next = start[1:]
    is_sorted = np.all(start_pre <= start_next)
    print(chrom, len(start), is_sorted)

1 3154 True
10 1230 True
11 1921 True
12 1765 True
13 666 True
14 1385 True
15 1135 True
16 1531 True
17 1894 True
18 662 True
19 2004 True
2 2285 True
20 895 True
21 502 True
22 837 True
3 1719 True
4 1353 True
5 1661 True
6 1632 True
7 1539 True
8 1320 True
9 1228 True
GL000009.2 1 True
GL000009.2 1 True
GL000194.1 2 True
GL000195.1 2 True
GL000205.2 1 True
GL000205.2 1 True
GL000213.1 1 True
GL000213.1 1 True
GL000218.1 1 True
GL000218.1 1 True
GL000219.1 1 True
GL000219.1 1 True
KI270711.1 1 True
KI270711.1 1 True
KI270713.1 2 True
KI270721.1 1 True
KI270721.1 1 True
KI270726.1 2 True
KI270727.1 4 True
KI270728.1 6 True
KI270731.1 1 True
KI270731.1 1 True
KI270734.1 3 True
MT 13 True
X 1078 True
Y 100 True


### Check illegal characters in gene name

In [8]:
# 0-9   
# 'A' - 'Z'
# 'a' - 'z'
# '-', '.', '_'
legal_ascii_int = list(range(48, 58)) + \
    list(range(65, 91)) +  \
    list(range(97, 123)) + \
    [45, 46, 95]
legal_chars = set([chr(i) for i in legal_ascii_int])

In [9]:
def contains_any_illegal_chars(x, chars):
    illegal_chars = [s for s in x if s not in chars]
    return(len(illegal_chars) > 0)

In [10]:
flag = dat[["gene_name"]].map(contains_any_illegal_chars, chars = legal_chars).values
dat[flag]

,gene_id,chrom,start,end,strand,gene_name,hgnc_id,gene_type


In [11]:
dat = dat[~flag]
dat

,gene_id,chrom,start,end,strand,gene_name,hgnc_id,gene_type
0,ENSG00000243485,1,29554,31109,+,MIR1302-2HG,NaN,lincRNA
1,ENSG00000237613,1,34554,36081,-,FAM138A,NaN,lincRNA
2,ENSG00000186092,1,65419,71585,+,OR4F5,NaN,protein_coding
3,ENSG00000238009,1,89295,133723,-,AL627309.1,NaN,lincRNA
4,ENSG00000239945,1,89551,91105,-,AL627309.3,NaN,lincRNA
...,...,...,...,...,...,...,...,...
33533,ENSG00000277856,KI270726.1,26241,26534,+,AC233755.2,NaN,protein_coding
33534,ENSG00000275063,KI270726.1,41444,41876,+,AC233755.1,NaN,protein_coding
33535,ENSG00000271254,KI270711.1,4612,29626,-,AC240274.1,NaN,protein_coding
33536,ENSG00000277475,KI270713.1,31698,32528,-,AC213203.1,NaN,protein_coding


### Check duplicate gene names

In [12]:
dup = dat[dat["gene_name"].duplicated(keep = False)].copy()
dup

,gene_id,chrom,start,end,strand,gene_name,hgnc_id,gene_type
2230,ENSG00000143248,1,163111121,163321791,-,RGS5,NaN,protein_coding
2232,ENSG00000232995,1,163244505,163321894,-,RGS5,NaN,antisense
2997,ENSG00000285053,1,235328570,235448952,+,TBCE,NaN,protein_coding
2999,ENSG00000284770,1,235367360,235452443,+,TBCE,NaN,protein_coding
4798,ENSG00000128655,2,177623252,178072755,-,PDE11A,NaN,protein_coding
4799,ENSG00000284741,2,177628069,178108339,-,PDE11A,NaN,protein_coding
5435,ENSG00000237940,2,241970683,241977276,+,LINC01238,NaN,lincRNA
5438,ENSG00000261186,2,242087351,242088457,-,LINC01238,NaN,antisense
5832,ENSG00000283706,3,46712115,46717907,-,PRSS50,NaN,protein_coding
5833,ENSG00000206549,3,46712115,46812574,-,PRSS50,NaN,protein_coding


In [13]:
dat = dat[~dat["gene_name"].duplicated(keep = "first")]
dat

,gene_id,chrom,start,end,strand,gene_name,hgnc_id,gene_type
0,ENSG00000243485,1,29554,31109,+,MIR1302-2HG,NaN,lincRNA
1,ENSG00000237613,1,34554,36081,-,FAM138A,NaN,lincRNA
2,ENSG00000186092,1,65419,71585,+,OR4F5,NaN,protein_coding
3,ENSG00000238009,1,89295,133723,-,AL627309.1,NaN,lincRNA
4,ENSG00000239945,1,89551,91105,-,AL627309.3,NaN,lincRNA
...,...,...,...,...,...,...,...,...
33533,ENSG00000277856,KI270726.1,26241,26534,+,AC233755.2,NaN,protein_coding
33534,ENSG00000275063,KI270726.1,41444,41876,+,AC233755.1,NaN,protein_coding
33535,ENSG00000271254,KI270711.1,4612,29626,-,AC240274.1,NaN,protein_coding
33536,ENSG00000277475,KI270713.1,31698,32528,-,AC213203.1,NaN,protein_coding


In [14]:
out_fn = os.path.join(out_dir, "refdata-cellranger-GRCh38-3.0.0.valid_name.uniq_name.8column.genes.unsort.tsv")
dat.to_csv(out_fn, sep = "\t", header = False, index = False)

### Subset genes in chr1-22,X,Y

In [15]:
chrom_list = ["%s" % i for i in range(1, 23)] + ["X", "Y"]
dat_subset = dat[dat["chrom"].isin(chrom_list)].copy()
dat_subset

,gene_id,chrom,start,end,strand,gene_name,hgnc_id,gene_type
0,ENSG00000243485,1,29554,31109,+,MIR1302-2HG,NaN,lincRNA
1,ENSG00000237613,1,34554,36081,-,FAM138A,NaN,lincRNA
2,ENSG00000186092,1,65419,71585,+,OR4F5,NaN,protein_coding
3,ENSG00000238009,1,89295,133723,-,AL627309.1,NaN,lincRNA
4,ENSG00000239945,1,89551,91105,-,AL627309.3,NaN,lincRNA
...,...,...,...,...,...,...,...,...
33491,ENSG00000160298,21,46300181,46323875,-,C21orf58,NaN,protein_coding
33492,ENSG00000160299,21,46324122,46445769,+,PCNT,NaN,protein_coding
33493,ENSG00000160305,21,46458899,46569852,+,DIP2A,NaN,protein_coding
33494,ENSG00000160307,21,46598962,46605208,-,S100B,NaN,protein_coding


In [16]:
dat_subset['chrom'] = pd.Categorical(dat_subset['chrom'], categories = chrom_list, ordered = True)
dat_subset = dat_subset.sort_values(by = ['chrom', 'start', 'end'])
dat_subset

,gene_id,chrom,start,end,strand,gene_name,hgnc_id,gene_type
0,ENSG00000243485,1,29554,31109,+,MIR1302-2HG,NaN,lincRNA
1,ENSG00000237613,1,34554,36081,-,FAM138A,NaN,lincRNA
2,ENSG00000186092,1,65419,71585,+,OR4F5,NaN,protein_coding
3,ENSG00000238009,1,89295,133723,-,AL627309.1,NaN,lincRNA
4,ENSG00000239945,1,89551,91105,-,AL627309.3,NaN,lincRNA
...,...,...,...,...,...,...,...,...
32152,ENSG00000228296,Y,25063083,25099892,-,TTTY4C,NaN,lincRNA
32153,ENSG00000223641,Y,25183643,25184773,-,TTTY17C,NaN,lincRNA
32154,ENSG00000228786,Y,25378300,25394719,-,LINC00266-4P,NaN,lincRNA
32155,ENSG00000172288,Y,25622162,25624902,+,CDY1,NaN,protein_coding


In [17]:
out_fn = os.path.join(out_dir, "refdata-cellranger-GRCh38-3.0.0.valid_name.uniq_name.chrom1-22XY.5column.genes.tsv")
subset_columns = ["chrom", "start", "end", "gene_name", "strand"]
dat_subset[subset_columns].to_csv(out_fn, sep = "\t", header = False, index = False)

### Subset genes in chr1-22

In [18]:
chrom_list = ["%s" % i for i in range(1, 23)]
df = dat_subset[dat_subset["chrom"].isin(chrom_list)].copy()

df['chrom'] = pd.Categorical(df['chrom'], categories = chrom_list, ordered = True)
df = df.sort_values(by = ['chrom', 'start', 'end'])
df

,gene_id,chrom,start,end,strand,gene_name,hgnc_id,gene_type
0,ENSG00000243485,1,29554,31109,+,MIR1302-2HG,NaN,lincRNA
1,ENSG00000237613,1,34554,36081,-,FAM138A,NaN,lincRNA
2,ENSG00000186092,1,65419,71585,+,OR4F5,NaN,protein_coding
3,ENSG00000238009,1,89295,133723,-,AL627309.1,NaN,lincRNA
4,ENSG00000239945,1,89551,91105,-,AL627309.3,NaN,lincRNA
...,...,...,...,...,...,...,...,...
32989,ENSG00000251322,22,50674415,50733298,+,SHANK3,NaN,protein_coding
32990,ENSG00000225929,22,50735825,50738139,-,AC000036.1,NaN,antisense
32991,ENSG00000100312,22,50738196,50745334,+,ACR,NaN,protein_coding
32992,ENSG00000254499,22,50740593,50743520,-,AC002056.2,NaN,antisense


In [19]:
out_fn = os.path.join(out_dir, "refdata-cellranger-GRCh38-3.0.0.valid_name.uniq_name.chrom1-22.5column.genes.tsv")
subset_columns = ["chrom", "start", "end", "gene_name", "strand"]
df[subset_columns].to_csv(out_fn, sep = "\t", header = False, index = False)

In [20]:
out_fn = os.path.join(out_dir, "refdata-cellranger-GRCh38-3.0.0.valid_name.uniq_name.chrom1-22.4column.genes.tsv")
subset_columns = ["chrom", "start", "end", "gene_name"]
df[subset_columns].to_csv(out_fn, sep = "\t", header = False, index = False)